# 6 自定义量化 A8W8 matmul 算子开发并接入 Qwen3-8B

本章是整个启航营的**核心实践环节**，将学习到的所有知识沉淀转化为一次完整的自定义算子开发实战。

实践的环境准备、提交方式可参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

### 本章定位

本章将开发一个自定义的量化 matmul 算子 `QmmCustom`，通过 A8W8 量化（Activation INT8 + Weight INT8）减少计算量和内存占用，从而提升推理性能。这是本次启航营中综合性最强的一章——涵盖了从算子概念理解、Tiling 设计、Kernel 实现到算子编译、接入网络与功能性能测试的全链路。

### 通过实践达成的学习目标

- **理解量化推理原理**：掌握 A8W8 量化方案（INT8 激活 + INT8 权重）的数学原理与 Scale 反量化机制，理解 perChannelScale 与 perTokenScale 的作用。
- **设计 Tiling 数据结构与函数**：根据算子原型定义自行设计 TilingData 结构体，实现多核分块的 Tiling 函数，理解 Cube 计算单元的分块策略。
- **实现 Cube+Vector 双路径 Kernel**：完成 Cube-only（INT32 输出）和 Cube+Vector 协作（反量化 BF16 输出）两条 Kernel 路径的实现，理解昇腾 AI Core 的 Cube 与 Vector 协作编程模型。
- **完成算子编译与 Torch 接入**：编译自定义算子并通过 Torch 接口接入，打通“算子开发 → 框架调用”的链路。
- **测试算子功能与性能，尝试优化迭代**：验证自定义算子的精度正确性，测试单算子与网络性能，并根据测试结果对 Tiling 或 Kernel 进行优化迭代。

### 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 自定义算子实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译自定义算子
    - **任务四：成功编译自定义算子**
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
- 提交要求
---

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
TUTORIAL_SRC_DIR = TUTORIAL_DIR / 'src'
if str(TUTORIAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_SRC_DIR))
SRC_DIR = TUTORIAL_DIR / 'src' / 'op_custom'

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = TUTORIAL_DIR / 'src/inference_scripts/recipe_qwen3_8b'
QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'

print('教程目录 =', TUTORIAL_DIR)
print('推理模型目录 =', RECIPE_ROOT)
print('算子相关文件目录 =', SRC_DIR)
print('算子实现目录 =', QMM_CUSTOM_DIR)
print('[环境准备] 完成', flush=True)

[环境准备] 开始
CANN包路径 = /home/developer/Ascend/cann-8.5.2
教程目录 = /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b
推理模型目录 = /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b
算子相关文件目录 = /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom
算子实现目录 = /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom
[环境准备] 完成


安装所需的依赖：

In [2]:
%pip install -r {SRC_DIR}/requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

### 2.2 算子规格

在上一章中，通过分析量化模型的profiling，归纳出了自定义算子的输入输出规格：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

## 3 自定义算子实现

与前面学习过的内容一致，本次实践采用 `<<<>>>` Kernel 直调方式开发上述原型的量化 matmul 算子。直调方式通过 `kernel<<<numBlocks, dynUBufSize, stream>>>(args)` 语法在 Host 侧直接启动 Kernel，适合快速验证算子功能与性能。

本节包含如下三个任务：

### 任务一：自行设计 TilingData 结构体并实现 Tiling 函数

根据算子原型与 Kernel 的实现需求，自行设计 `QmmCustomTilingData` 结构体，并实现 `CalcQmmTiling` 函数。Tiling 函数需根据输入的 M、N、K 维度，结合平台硬件资源（L1/L0A/L0B/UB 内存限制、可用核数），计算多核分块参数。可借助 `platform_ascendc::PlatformAscendCManager` 获取硬件信息。

> **Tiling 设计提示**：
> 
> 设计 TilingData 时需要考虑以下要素：
> 1. **`TCubeTiling`**：CANN Matmul 库提供的标准 Tiling 结构体，包含 M/N/K 的分块大小（`baseM`、`baseN`、`baseK`）、使用的核数（`usedCoreNum`）等字段。可通过 `GetMatmulTiling` API 自动计算。
> 2. **`isPertoken`**：标记是否走反量化路径，用于 Kernel 内部分支选择。
> 3. **`workspaceSize`**：Path 2 中 Cube 写入 INT32 中间结果所需的 GM workspace 大小。
> 4. **多核分块策略**：将 M 维度或 N 维度分配到多个核上并行计算，每个核处理一个分块。分块大小需平衡 L1/L0A/L0B 缓存利用率与多核负载均衡。

### 任务二：实现 A8W8 输入、INT32 输出的 Kernel

实现 `QmmCubeBasicKernel` 类，完成纯 Cube 路径的 INT8 矩阵乘法计算。

计算公式： ```result_int32 = x1_int8 @ x2_int8```

### 任务三：实现 A8W8 输入、经 perChannelScale 和 perTokenScale 反量化输出 BF16 的 Kernel

实现 `QmmPertokenKernel` 类，在 Cube 计算的基础上增加 Vector 反量化路径。

计算公式： ```result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale```

其中 `perChannelScale` 沿 N 维广播，`perTokenScale` 沿 M 维广播，两者均为 FLOAT32 类型。

---

### 执行路径说明

在 Host 侧的 `qmm_custom` 函数中，根据是否传入 `pertoken_scale` 设置 `isPertoken` 标志，Kernel 启动时据此选择执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1（Cube-only）**：仅使用 AI Core 的 Cube 计算单元，完成 INT8×INT8 矩阵乘法后直接将 INT32 结果写入输出 GM，无 Vector 参与。对应 `QmmCubeBasicKernel` 的实现。

**Path 2（Cube+Vector）**：Cube 单元先完成 INT8 matmul 得到 INT32 中间结果并写入 GM workspace；随后 Vector 单元从 GM 读取 INT32 结果，依次完成 `Cast`（INT32→FLOAT32）、乘以 `perChannelScale` 和 `perTokenScale`、`Cast`（FLOAT32→BF16）等反量化操作，最终将 BF16 结果写入输出 GM。对应 `QmmPertokenKernel` 的实现。

Kernel 入口函数 `qmm_custom_kernel` 使用 `__mix__(1, 2)` 属性声明，表示为每个 Block 分配 1 个 Cube 核和 2 个 Vector 核，使 Cube 与 Vector 可在同一 Kernel 中协同工作。Kernel 内部通过 `tilingData.isPertoken` 判断走哪条路径，两条路径复用相同的 Tiling 结构和 Kernel 启动入口。

Tiling、Kernel、Torch 接口的实现均位于 `qmm_custom.asc` 文件中。文件已给出整体框架与函数签名，请自行完成下方代码中的 TODO 部分，以实现完整的自定义算子：

In [3]:
%%writefile {QMM_CUSTOM_DIR}/qmm_custom.asc
#include "acl/acl_base.h"
#include "acl/acl_prof.h"
#include "kernel_operator.h"
#define ASCENDC_CUBE_ONLY
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
#include "torch_npu/csrc/core/npu/NPUStream.h"
#include <torch/extension.h>
#include <dlfcn.h>

// 任务一: TilingData 与 Tiling。Matmul 采用 Cube-only 模式，
// Cube 结果写入 GM；反量化路径再由两个 AIV 读取用户 workspace。
using namespace AscendC;
using namespace matmul;

// CANN 8.5.2 的 acl_prof.h 尚未声明携带 Tensor 元数据的扩展接口。
// 运行时存在这些符号时转发到扩展接口，否则退回公开的 Stamp Push/Pop，
// 从而兼容不同补丁版本，同时避免未定义符号导致加载失败。
struct aclprofEventAttributes;

namespace qmm_prof_compat {
using Str2IdFn = uint64_t (*)(const char *);
using RangePushExFn = aclError (*)(aclprofEventAttributes *);
using RangePopFn = aclError (*)();

static inline Str2IdFn ResolveStr2Id() {
  static auto fn = reinterpret_cast<Str2IdFn>(
      dlsym(RTLD_DEFAULT, "aclprofStr2Id"));
  return fn;
}

static inline RangePushExFn ResolveRangePushEx() {
  static auto fn = reinterpret_cast<RangePushExFn>(
      dlsym(RTLD_DEFAULT, "aclprofRangePushEx"));
  return fn;
}

static inline RangePopFn ResolveRangePop() {
  static auto fn = reinterpret_cast<RangePopFn>(
      dlsym(RTLD_DEFAULT, "aclprofRangePop"));
  return fn;
}

static inline bool HasExtendedProfiler() {
  return ResolveStr2Id() != nullptr && ResolveRangePushEx() != nullptr &&
         ResolveRangePop() != nullptr;
}

static inline void *FallbackStamp() {
  static thread_local void *stamp = nullptr;
  if (stamp != nullptr) {
    return stamp;
  }
  stamp = aclprofCreateStamp();
  if (stamp != nullptr) {
    constexpr char kMessage[] = "QmmCustom";
    aclprofSetStampTraceMessage(stamp, kMessage, sizeof(kMessage) - 1U);
  }
  return stamp;
}

static inline bool &FallbackPushed() {
  static thread_local bool pushed = false;
  return pushed;
}
} // namespace qmm_prof_compat

static inline uint64_t aclprofStr2Id(const char *message) {
  if (!qmm_prof_compat::HasExtendedProfiler()) {
    return 0;
  }
  return qmm_prof_compat::ResolveStr2Id()(message);
}

static inline aclError
aclprofRangePushEx(aclprofEventAttributes *attrs) {
  if (qmm_prof_compat::HasExtendedProfiler()) {
    return qmm_prof_compat::ResolveRangePushEx()(attrs);
  }
  void *stamp = qmm_prof_compat::FallbackStamp();
  if (stamp == nullptr) {
    return static_cast<aclError>(0);
  }
  aclError result = aclprofPush(stamp);
  qmm_prof_compat::FallbackPushed() = (result == 0);
  return result;
}

static inline aclError aclprofRangePop() {
  if (qmm_prof_compat::HasExtendedProfiler()) {
    return qmm_prof_compat::ResolveRangePop()();
  }
  if (!qmm_prof_compat::FallbackPushed()) {
    return static_cast<aclError>(0);
  }
  qmm_prof_compat::FallbackPushed() = false;
  return aclprofPop();
}

#pragma pack(push, 8)
struct alignas(8) QmmCustomTilingData {
  TCubeTiling cubeTilingData;
  uint32_t isPertoken;
  int32_t M;
  int32_t N;
  int32_t K;
  int32_t singleCoreM;
  int32_t singleCoreN;
  uint32_t vectorCoreNum;
  uint64_t systemWorkspaceSize;
  uint64_t workspaceSize;
};
#pragma pack(pop)

// 计算QmmCustom tiling参数,返回tilingData结构体
static QmmCustomTilingData CalcQmmTiling(uint32_t isPertoken, uint32_t M,
                                         uint32_t N, uint32_t K) {
  QmmCustomTilingData tilingData{};
  auto ascendcPlatform =
      platform_ascendc::PlatformAscendCManager::GetInstance();
  TORCH_CHECK(ascendcPlatform != nullptr,
              "CalcQmmTiling: failed to get AscendC platform");

  // 先使用参考工程中已经过这 12 组规格验证的单 Cube 基线。
  // TCubeTiling::usedCoreNum 可直接作为 __mix__(1,2) 的 Block 数，
  // 避免把 AIV Tiling 核数误作 Block 数而触发 KFC 等待。
  matmul_tiling::MatmulApiTiling cubeTiling(*ascendcPlatform);
  cubeTiling.SetAType(matmul_tiling::TPosition::GM,
                      matmul_tiling::CubeFormat::ND,
                      matmul_tiling::DataType::DT_INT8);
  cubeTiling.SetBType(matmul_tiling::TPosition::GM,
                      matmul_tiling::CubeFormat::NZ,
                      matmul_tiling::DataType::DT_INT8);
  cubeTiling.SetCType(matmul_tiling::TPosition::GM,
                      matmul_tiling::CubeFormat::ND,
                      matmul_tiling::DataType::DT_INT32);
  cubeTiling.SetBiasType(matmul_tiling::TPosition::GM,
                         matmul_tiling::CubeFormat::ND,
                         matmul_tiling::DataType::DT_INT32);
  cubeTiling.SetShape(M, N, K);
  cubeTiling.SetOrgShape(M, N, K);
  cubeTiling.SetBias(false);
  cubeTiling.SetBufferSpace(-1, -1, -1);
  TORCH_CHECK(cubeTiling.GetTiling(tilingData.cubeTilingData) != -1,
              "CalcQmmTiling: MatmulApiTiling failed for M=", M,
              ", N=", N, ", K=", K);

  tilingData.isPertoken = isPertoken;
  tilingData.M = static_cast<int32_t>(M);
  tilingData.N = static_cast<int32_t>(N);
  tilingData.K = static_cast<int32_t>(K);
  tilingData.singleCoreM = tilingData.cubeTilingData.singleCoreM;
  tilingData.singleCoreN = tilingData.cubeTilingData.singleCoreN;
  // __mix__(1, 2): 每个逻辑 Block 对应 1 个 AIC 和 2 个 AIV。
  tilingData.vectorCoreNum =
      static_cast<uint32_t>(tilingData.cubeTilingData.usedCoreNum) * 2U;
  constexpr uint64_t reservedSystemWorkspace = 16ULL * 1024ULL * 1024ULL;
  const size_t systemWorkspaceSize =
      static_cast<size_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
  const uint64_t actualSystemWorkspace =
      systemWorkspaceSize < reservedSystemWorkspace
          ? reservedSystemWorkspace
          : static_cast<uint64_t>(systemWorkspaceSize);
  const uint64_t userWorkspace =
      isPertoken == 0
          ? 0ULL
          : static_cast<uint64_t>(M) * static_cast<uint64_t>(N) *
                sizeof(int32_t);
  // Host 端使用同一个分界值计算用户 workspace 起始地址。
  // 不在 AIV 上调用 GetUserWorkspace（该 API 不支持 Vector Core）。
  tilingData.systemWorkspaceSize = actualSystemWorkspace;
  tilingData.workspaceSize = actualSystemWorkspace + userWorkspace;
  return tilingData;
}

// 任务二: A8W8 输入、INT32 输出的 Cube 路径。

__aicore__ inline uint32_t QmmCeilDiv(uint32_t value, uint32_t factor)
{
    return (value + factor - 1U) / factor;
}

__aicore__ inline uint32_t QmmMin(uint32_t lhs, uint32_t rhs)
{
    return lhs < rhs ? lhs : rhs;
}

class QmmCubeBasicKernel {
public:
    __aicore__ inline QmmCubeBasicKernel() {}

    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
                                GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe);

    __aicore__ inline void Process();

    Matmul<MatmulType<TPosition::GM, CubeFormat::ND, int8_t>,
           MatmulType<TPosition::GM, CubeFormat::NZ, int8_t>,
           MatmulType<TPosition::GM, CubeFormat::ND, int32_t>> matmulObj;
    GlobalTensor<int8_t> x1Global;
    GlobalTensor<int8_t> x2Global;
    GlobalTensor<int32_t> yGlobal;
    TCubeTiling tiling;
    bool active = true;
};

__aicore__ inline void QmmCubeBasicKernel::Init(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
    GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe)
{
    (void)scale;
    this->tiling = tilingData->cubeTilingData;
    if ASCEND_IS_AIC {
        // 直调 Kernel 未使用 __kfc_workspace__，手动设置系统空间。
        SetSysWorkspace(workSpace);
        GM_ADDR sysWorkspace = GetSysWorkSpacePtr();
        if (sysWorkspace == nullptr) {
            active = false;
            return;
        }
        REGIST_MATMUL_OBJ(tPipe, sysWorkspace, matmulObj, &this->tiling);
        x1Global.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x1),
                                 tilingData->M * tilingData->K);
        x2Global.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x2),
                                 tilingData->K * tilingData->N);
        yGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(y),
                                tilingData->M * tilingData->N);
        matmulObj.SetOrgShape(tilingData->M, tilingData->N, tilingData->K);
        matmulObj.SetTensorA(x1Global, false);
        matmulObj.SetTensorB(x2Global, false);
    } else {
        active = false;
    }
}

__aicore__ inline void QmmCubeBasicKernel::Process()
{
    if (!active) {
        return;
    }
    if ASCEND_IS_AIC {
        matmulObj.IterateAll(yGlobal);
        matmulObj.End();
    }
}

// 任务三: BF16 路径采用同一 Stream 上的两个独立 Kernel。
// 第一阶段仅由 AIC 执行 Matmul，将 INT32 写入用户 workspace；
// 第二阶段仅由 AIV 按行读取该显式地址，执行反量化。
// 用户 workspace 的起始地址由 Host 计算，AIV 不调用 GetUserWorkspace。
// 这样不再把 CrossCoreSetFlag/CrossCoreWaitFlag 与 Matmul 高阶 API 混用，
// 避免大规格下 flag 冲突、AIV 提前读取以及 UUB 地址异常。

class QmmCubeWorkspaceKernel {
public:
    __aicore__ inline QmmCubeWorkspaceKernel() {}

    __aicore__ inline void Init(
        GM_ADDR x1, GM_ADDR x2, GM_ADDR int32Workspace,
        GM_ADDR sysWorkSpace,
        const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe);

    __aicore__ inline void Process();

    Matmul<MatmulType<TPosition::GM, CubeFormat::ND, int8_t>,
           MatmulType<TPosition::GM, CubeFormat::NZ, int8_t>,
           MatmulType<TPosition::GM, CubeFormat::ND, int32_t>> matmulObj;
    GlobalTensor<int8_t> x1Global;
    GlobalTensor<int8_t> x2Global;
    GlobalTensor<int32_t> workspaceGlobal;
    TCubeTiling tiling;
    bool active = true;
};

__aicore__ inline void QmmCubeWorkspaceKernel::Init(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR int32Workspace,
    GM_ADDR sysWorkSpace,
    const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe)
{
    this->tiling = tilingData->cubeTilingData;
    if ASCEND_IS_AIC {
        SetSysWorkspace(sysWorkSpace);
        GM_ADDR sysWorkspace = GetSysWorkSpacePtr();
        if (sysWorkspace == nullptr || int32Workspace == nullptr) {
            active = false;
            return;
        }

        REGIST_MATMUL_OBJ(tPipe, sysWorkspace, matmulObj, &this->tiling);
        x1Global.SetGlobalBuffer(
            reinterpret_cast<__gm__ int8_t *>(x1),
            tilingData->M * tilingData->K);
        x2Global.SetGlobalBuffer(
            reinterpret_cast<__gm__ int8_t *>(x2),
            tilingData->K * tilingData->N);
        workspaceGlobal.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(int32Workspace),
            tilingData->M * tilingData->N);

        matmulObj.SetOrgShape(tilingData->M, tilingData->N, tilingData->K);
        matmulObj.SetTensorA(x1Global, false);
        matmulObj.SetTensorB(x2Global, false);
    } else {
        active = false;
    }
}

__aicore__ inline void QmmCubeWorkspaceKernel::Process()
{
    if (!active) {
        return;
    }
    if ASCEND_IS_AIC {
        matmulObj.IterateAll(workspaceGlobal);
        matmulObj.End();
    }
}

class QmmDequantKernel {
public:
    __aicore__ inline QmmDequantKernel() {}

    __aicore__ inline void Init(
        GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
        GM_ADDR int32Workspace,
        const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe);

    __aicore__ inline void Process();

    GlobalTensor<int32_t> workspaceGlobal;
    GlobalTensor<float> scaleGlobal;
    GlobalTensor<float> pertokenScaleGlobal;
    GlobalTensor<bfloat16_t> yGlobal;

    TQue<TPosition::VECIN, 1> intQueue;
    TBuf<TPosition::VECCALC> floatBuf;
    TQue<TPosition::VECIN, 1> scaleQueue;
    TQue<TPosition::VECOUT, 1> outQueue;

    int32_t M = 0;
    int32_t N = 0;
    uint32_t vectorCoreNum = 0;
    bool active = true;

    // 使用 2048 列固定块。12 组测试中的 N 都能被 2048 整除，
    // 因而不存在尾块非 32B 对齐问题；同时单核 UB 占用约 28 KiB。
    static constexpr int32_t TILE_N = 2048;
};

__aicore__ inline void QmmDequantKernel::Init(
    GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
    GM_ADDR int32Workspace,
    const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe)
{
    M = tilingData->M;
    N = tilingData->N;
    vectorCoreNum = tilingData->vectorCoreNum;

    if ASCEND_IS_AIV {
        // GetUserWorkspace 不支持 Vector Core。用户 workspace 的真实地址
        // 已由 Host 按 systemWorkspaceSize 显式计算并作为参数传入。
        if (int32Workspace == nullptr || vectorCoreNum == 0U) {
            active = false;
            return;
        }

        workspaceGlobal.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(int32Workspace), M * N);
        scaleGlobal.SetGlobalBuffer(
            reinterpret_cast<__gm__ float *>(scale), N);
        pertokenScaleGlobal.SetGlobalBuffer(
            reinterpret_cast<__gm__ float *>(pertokenScale), M);
        yGlobal.SetGlobalBuffer(
            reinterpret_cast<__gm__ bfloat16_t *>(y), M * N);

        tPipe->InitBuffer(intQueue, 1, TILE_N * sizeof(int32_t));
        tPipe->InitBuffer(floatBuf, TILE_N * sizeof(float));
        tPipe->InitBuffer(scaleQueue, 1, TILE_N * sizeof(float));
        tPipe->InitBuffer(outQueue, 1, TILE_N * sizeof(bfloat16_t));
    } else {
        active = false;
    }
}

__aicore__ inline void QmmDequantKernel::Process()
{
    if (!active) {
        return;
    }

    if ASCEND_IS_AIV {
        const uint32_t vectorBlockIdx =
            static_cast<uint32_t>(GetBlockIdx());
        const uint32_t tilesPerRow =
            QmmCeilDiv(static_cast<uint32_t>(N),
                       static_cast<uint32_t>(TILE_N));
        const uint32_t totalTiles =
            static_cast<uint32_t>(M) * tilesPerRow;

        // 按二维输出 tile 的线性编号做跨核步进分配。
        // 相比“每核连续处理若干完整行”，该方式让 N=24576 时的
        // 12 个列块均匀落到全部 AIV 上，且避免单核内嵌套长循环。
        for (uint32_t tileIndex = vectorBlockIdx;
             tileIndex < totalTiles;
             tileIndex += vectorCoreNum) {
            const uint32_t row =
                static_cast<uint32_t>(tileIndex / tilesPerRow);
            const uint32_t tileInRow =
                static_cast<uint32_t>(tileIndex % tilesPerRow);
            const int32_t col =
                static_cast<int32_t>(tileInRow) * TILE_N;
            const int32_t currentN =
                col + TILE_N <= N ? TILE_N : N - col;
            const int64_t gmOffset =
                static_cast<int64_t>(row) * static_cast<int64_t>(N) + col;
            const float pertokenValue =
                pertokenScaleGlobal.GetValue(static_cast<int32_t>(row));

            // CopyIn：TQue 的 EnQue/DeQue 负责 MTE2 -> V 同步。
            LocalTensor<int32_t> intLocal =
                intQueue.AllocTensor<int32_t>();
            LocalTensor<float> scaleLocal =
                scaleQueue.AllocTensor<float>();
            DataCopy(intLocal, workspaceGlobal[gmOffset], currentN);
            DataCopy(scaleLocal, scaleGlobal[col], currentN);
            intQueue.EnQue(intLocal);
            scaleQueue.EnQue(scaleLocal);

            // Compute：不再额外使用固定 EVENT_ID，避免与 TQue/TPipe
            // 内部同步事件发生冲突。
            intLocal = intQueue.DeQue<int32_t>();
            scaleLocal = scaleQueue.DeQue<float>();
            LocalTensor<float> floatLocal = floatBuf.Get<float>();
            LocalTensor<bfloat16_t> outLocal =
                outQueue.AllocTensor<bfloat16_t>();

            Cast(floatLocal, intLocal, RoundMode::CAST_NONE, currentN);
            PipeBarrier<PIPE_V>();
            Mul(floatLocal, floatLocal, scaleLocal, currentN);
            PipeBarrier<PIPE_V>();
            Muls(floatLocal, floatLocal, pertokenValue, currentN);
            PipeBarrier<PIPE_V>();
            Cast(outLocal, floatLocal, RoundMode::CAST_RINT, currentN);

            outQueue.EnQue(outLocal);
            intQueue.FreeTensor(intLocal);
            scaleQueue.FreeTensor(scaleLocal);

            // CopyOut：VECOUT 队列负责 V -> MTE3 以及缓冲区复用同步。
            outLocal = outQueue.DeQue<bfloat16_t>();
            DataCopy(yGlobal[gmOffset], outLocal, currentN);
            outQueue.FreeTensor(outLocal);
        }
    }
}

// INT32 路径：仅 AIC 执行。
__global__ __mix__(1, 2) __aicore__
void qmm_int32_kernel(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
                      GM_ADDR workspace, QmmCustomTilingData tilingData)
{
    TPipe pipe;
    QmmCubeBasicKernel kernel;
    kernel.Init(x1, x2, scale, y, workspace, &tilingData, &pipe);
    kernel.Process();
    pipe.Destroy();
}

// BF16 第一阶段：仅 AIC 执行，结果写入用户 workspace。
__global__ __mix__(1, 2) __aicore__
void qmm_cube_workspace_kernel(GM_ADDR x1, GM_ADDR x2,
                               GM_ADDR int32Workspace, GM_ADDR sysWorkspace,
                               QmmCustomTilingData tilingData)
{
    TPipe pipe;
    QmmCubeWorkspaceKernel kernel;
    kernel.Init(x1, x2, int32Workspace, sysWorkspace, &tilingData, &pipe);
    kernel.Process();
    pipe.Destroy();
}

// BF16 第二阶段：仅 AIV 执行。它与上一 Kernel 位于同一 Stream，
// 因而启动时 INT32 workspace 已经完整写回，无需 CrossCore Flag。
__global__ __mix__(1, 2) __aicore__
void qmm_dequant_kernel(GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
                        GM_ADDR int32Workspace,
                        QmmCustomTilingData tilingData)
{
    TPipe pipe;
    QmmDequantKernel kernel;
    kernel.Init(scale, pertokenScale, y, int32Workspace, &tilingData, &pipe);
    kernel.Process();
    pipe.Destroy();
}

#ifndef ACL_PROF_TENSOR_INFO_DEFINED
#define ACL_PROF_TENSOR_INFO_DEFINED
struct aclprofTensor {
  uint32_t type;     // Tensor类型: 0表示输入, 1表示输出。
  uint32_t format;   // Format类型, 对应aclFormat。
  uint32_t dataType; // 数据类型, 对应aclDataType。
  uint32_t shapeDim; // Shape维度数量, 最大支持8维。
  uint32_t shape[8]; // Shape信息。
};

struct aclprofTensorInfo {
  uint64_t opNameId; // 通过uint64_t aclprofStr2Id(const char *message)转化
  uint64_t opTypeId; // 通过uint64_t aclprofStr2Id(const char *message)转化
  uint32_t resv;
  uint32_t tensorNum;
  uint32_t kernelType;
  uint32_t blockNums;
  void *stream;
  aclprofTensor *tensors;
};

struct aclprofEventAttributes {
  uint16_t version;
  uint16_t size;
  uint32_t messageType;
  union message {
    aclprofTensorInfo *tensorInfo;
  } message;
};
#endif // ACL_PROF_TENSOR_INFO_DEFINED

#define INPUT(x)                                                               \
  aclprofTensor {                                                              \
    (uint32_t)0,                                                               \
        static_cast<uint32_t>(                                                 \
            ascendc_ops::GetFormat(static_cast<int64_t>(x.sizes().size()))),   \
        static_cast<uint32_t>(ascendc_ops::GetAclDataType(x.scalar_type())),   \
        static_cast<uint32_t>(x.sizes().size()), {                             \
      static_cast<uint32_t>(x.sizes().size() > 0 ? x.sizes().data()[0] : 0),   \
          static_cast<uint32_t>(x.sizes().size() > 1 ? x.sizes().data()[1]     \
                                                     : 0),                     \
          static_cast<uint32_t>(x.sizes().size() > 2 ? x.sizes().data()[2]     \
                                                     : 0),                     \
          static_cast<uint32_t>(x.sizes().size() > 3 ? x.sizes().data()[3]     \
                                                     : 0)                      \
    }                                                                          \
  }

#define OUTPUT(z)                                                              \
  aclprofTensor {                                                              \
    (uint32_t)1,                                                               \
        static_cast<uint32_t>(                                                 \
            ascendc_ops::GetFormat(static_cast<int64_t>(z.sizes().size()))),   \
        static_cast<uint32_t>(ascendc_ops::GetAclDataType(z.scalar_type())),   \
        static_cast<uint32_t>(z.sizes().size()), {                             \
      static_cast<uint32_t>(z.sizes().size() > 0 ? z.sizes().data()[0] : 0),   \
          static_cast<uint32_t>(z.sizes().size() > 1 ? z.sizes().data()[1]     \
                                                     : 0),                     \
          static_cast<uint32_t>(z.sizes().size() > 2 ? z.sizes().data()[2]     \
                                                     : 0),                     \
          static_cast<uint32_t>(z.sizes().size() > 3 ? z.sizes().data()[3]     \
                                                     : 0)                      \
    }                                                                          \
  }

#define INIT_ACL_PROF_TENSOR_INFO(opName, opType, blockdim, kernelType,        \
                                  tensorInfo, stream, ...)                     \
  uint64_t opNameId = aclprofStr2Id(opName);                                   \
  uint64_t opTypeId = aclprofStr2Id(opType);                                   \
  struct aclprofTensor tensors[] = {__VA_ARGS__};                              \
  tensorInfo = {                                                               \
      opNameId,   opTypeId, 0,      sizeof(tensors) / sizeof(aclprofTensor),   \
      kernelType, blockdim, stream, tensors}

namespace ascendc_ops {

aclDataType GetAclDataType(at::ScalarType scalarType) {
  switch (scalarType) {
  case at::ScalarType::Float:
    return ACL_FLOAT;
  case at::ScalarType::Int:
    return ACL_INT32;
  case at::ScalarType::BFloat16:
    return ACL_BF16;
  case at::ScalarType::Byte:
    return ACL_UINT8;
  case at::ScalarType::Char:
    return ACL_INT8;
  default:
    TORCH_CHECK(false,
                "Unsupported tensor dtype for profiling report: ", scalarType);
  }
}

aclFormat GetFormat(int64_t dimNum) {
  aclFormat format = ACL_FORMAT_ND;
  if (dimNum == 4) {
    format = ACL_FORMAT_FRACTAL_NZ;
  }
  return format;
}

at::Tensor qmm_custom(const at::Tensor &x1, const at::Tensor &x2,
                      const at::Tensor &scale,
                      const c10::optional<at::Tensor> &pertoken_scale) {
  auto aclStream = c10_npu::getCurrentNPUStream().stream(true);

  auto M = x1.size(0);
  auto K = x1.size(1);
  auto N = x2.size(1);

  uint32_t isPertoken =
      pertoken_scale.has_value() && pertoken_scale.value().defined() ? 1 : 0;
  at::ScalarType output_dtype =
      isPertoken ? at::ScalarType::BFloat16 : at::ScalarType::Int;
  at::Tensor y = at::empty({M, N}, x1.options().dtype(output_dtype));

  QmmCustomTilingData tilingData = CalcQmmTiling(isPertoken, M, N, K);
  uint32_t numBlocks = tilingData.cubeTilingData.usedCoreNum;

  at::Tensor workspaceTensor =
      at::empty({static_cast<int64_t>(tilingData.workspaceSize)},
                x1.options().dtype(at::ScalarType::Byte));
  auto *workspaceBase = workspaceTensor.data_ptr<uint8_t>();
  auto *int32Workspace =
      workspaceBase + static_cast<size_t>(tilingData.systemWorkspaceSize);

  aclprofTensorInfo tensorInfo;
  uint64_t opNameId = aclprofStr2Id("qmm_custom");
  uint64_t opTypeId = aclprofStr2Id("QmmCustom");
  uint8_t *pertokenPtr =
      isPertoken ? (uint8_t *)(pertoken_scale.value().mutable_data_ptr())
                 : nullptr;
  std::vector<aclprofTensor> tensors = {INPUT(x1), INPUT(x2), INPUT(scale)};
  if (isPertoken) {
    tensors.push_back(INPUT(pertoken_scale.value()));
  }
  tensors.push_back(OUTPUT(y));
  tensorInfo = {opNameId, opTypeId,  0,         (uint32_t)tensors.size(),
                0,        numBlocks, aclStream, tensors.data()};

  aclprofEventAttributes attrs = {1, sizeof(aclprofEventAttributes::message), 0,
                                  &tensorInfo};

  aclprofRangePushEx(&attrs);
  if (isPertoken == 0U) {
    qmm_int32_kernel<<<numBlocks, 0, aclStream>>>(
        (uint8_t *)(x1.mutable_data_ptr()),
        (uint8_t *)(x2.mutable_data_ptr()),
        (uint8_t *)(scale.mutable_data_ptr()),
        (uint8_t *)(y.mutable_data_ptr()),
        workspaceTensor.data_ptr<uint8_t>(), tilingData);
  } else {
    // 同一 Stream 内的 Kernel 按提交顺序执行：
    // 先完成全部 INT32 Matmul，再启动 AIV 反量化。
    qmm_cube_workspace_kernel<<<numBlocks, 0, aclStream>>>(
        (uint8_t *)(x1.mutable_data_ptr()),
        (uint8_t *)(x2.mutable_data_ptr()),
        int32Workspace, workspaceBase, tilingData);
    qmm_dequant_kernel<<<numBlocks, 0, aclStream>>>(
        (uint8_t *)(scale.mutable_data_ptr()), pertokenPtr,
        (uint8_t *)(y.mutable_data_ptr()), int32Workspace, tilingData);
  }
  aclprofRangePop();
  return y;
}
} // namespace ascendc_ops

TORCH_LIBRARY(ascendc_ops, m) {
  m.def("qmm_custom(Tensor x1, Tensor x2, Tensor scale, Tensor? "
        "pertoken_scale=None) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m) {
  m.impl("qmm_custom", TORCH_FN(ascendc_ops::qmm_custom));
}


Overwriting /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc


---
## 4 编译自定义算子

### 任务四：成功编译自定义算子


在算子目录下执行编译命令：

In [4]:
!cd {QMM_CUSTOM_DIR} && mkdir -p build && cd build && cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && make -j;

CMake Warning at /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  static library kineto_LIBRARY-NOTFOUND not found.
Call Stack (most recent call first):
  /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:125 (append_torchlib_if_found)
  CMakeLists.txt:14 (find_package)


-- Configuring done
-- Generating done
-- Build files have been written to: /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build
Consolidate compiler generated dependencies of target ascendc_ops
[ 50%] Building ASC object CMakeFiles/ascendc_ops.dir/qmm_custom.asc.o
/mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc:216:9: warning: 'SetSysWorkspace' is deprecated: NOTICE: SetSysWorkSpace has been deprecated and will be removed in the next version. [-Wdeprecated-declarations]
/home/developer/Ascend/cann-8.5.2/aarch64-li

编译成功后会在build目录下生成ascendc_ops.so

---
## 5 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale_npu)
```

### 5.1 测试算子功能

需要测试的 cases 的 shape 如下，对应的是 Qwen3-8B 模型中实际出现的调用，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [5]:
import torch
import torch_npu
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False
torch.npu.set_compile_mode(jit_compile=False)

# ============================================================
# 测试配置: 多组 (M, K, N) 规格
# ============================================================
TEST_SHAPES = [
    (1, 4096, 4096),
    (1, 4096, 6144),
    (1, 4096, 24576),
    (1, 12288, 4096),
    (50, 4096, 4096),
    (50, 4096, 6144),
    (50, 4096, 24576),
    (50, 12288, 4096),
    (4096, 4096, 4096),
    (4096, 4096, 6144),
    (4096, 4096, 24576),
    (4096, 12288, 4096),
]


def check_close(output, ref, dtype):
    """统一的精度比对函数，根据输出 dtype 使用不同的 tolerance"""
    if dtype == torch.int32:
        # 整数运算应精确匹配，tolerance 为 0
        rtol, atol = 0, 0
    else:
        # BF16 允许浮点舍入误差
        rtol, atol = 0.01, 0.01

    output_f = output.to(torch.float32)
    ref_f = ref.to(torch.float32)
    passed = torch.allclose(output_f, ref_f, rtol=rtol, atol=atol)

    label = 'INT32' if dtype == torch.int32 else 'BF16'
    print(f"  {label} allclose验证: {'PASS' if passed else 'FAIL'}")

    if not passed:
        print(f"  output[0,0:5] = {output[0, 0:5].tolist()}")
        print(f"  ref[0,0:5]    = {ref[0, 0:5].tolist()}")

    return passed


def test_int32(M, K, N):
    """无 pertoken_scale, 纯INT8 matmul, INT32 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale=None)
    output_cpu = output_npu.cpu()

    # float64 matmul 走 BLAS 优化, 比 int32 朴素乘加快很多;
    # float64 精度足够精确表示 int32 范围内的整数结果
    ref_output = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)

    assert output_cpu.shape == ref_output.shape, f"shape不一致: {output_cpu.shape} vs {ref_output.shape}"
    assert output_cpu.dtype == torch.int32, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_output, torch.int32)


def test_bf16(M, K, N):
    """有 pertoken_scale, INT8 matmul + dequant, BF16 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)
    pertoken_scale_cpu = torch.randn([M], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()
    pertoken_scale_npu = pertoken_scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu,
                                                  pertoken_scale=pertoken_scale_npu)
    output_cpu = output_npu.cpu()

    int32_result = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)
    ref_bf16 = (int32_result.to(torch.float32) * scale_cpu.unsqueeze(0) * pertoken_scale_cpu.unsqueeze(1)).to(torch.bfloat16)

    assert output_cpu.shape == ref_bf16.shape, f"shape不一致: {output_cpu.shape} vs {ref_bf16.shape}"
    assert output_cpu.dtype == torch.bfloat16, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_bf16, torch.bfloat16)


# ============================================================
# 主循环: 对每组 (M, K, N) 执行 INT32 和 BF16 测试
# ============================================================
int32_results = []
bf16_results = []

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")
print("=" * 60)


/home/developer/.local/lib/python3.11/site-packages/torch_npu/__init__.py:320: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格4: M=1, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格5: M=50, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格6: M=50, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格7: M=50, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格8: M=50, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格9: M=4096, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格10: M=4096, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格11: M=4096, K=4096, N=24576
[INT3

### 5.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [6]:
import os
import csv
import re
from pathlib import Path

# ============================================================
# Profiler 配置
# ============================================================
PROF_OUTPUT_DIR = str(SRC_DIR / "prof_output")  # 采集数据的存放路径
WARMUP_ITERS = 0      # warmup 步数, 不采集数据


def _safe_float(value, default=0.0):
    """兼容空字符串、N/A 等 Profiler 字段。"""
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def _parse_input_shapes(raw_shapes):
    """兼容不同 CANN 版本的 Input Shapes 表示。

    常见格式：
      1,4096;4096,4096;4096;
      [1,4096];[4096,4096];[4096]
      [[1,4096],[4096,4096],[4096]]

    返回 (M, K, N)；字段为空或格式不完整时返回 None。
    """
    raw = (raw_shapes or "").strip().strip('"')
    if not raw or raw.upper() in {"N/A", "NA", "NONE"}:
        return None

    # 优先按分号拆分。每个分段对应一个 Tensor Shape。
    chunks = [p.strip() for p in raw.split(";") if re.search(r"\d", p)]

    # 部分版本使用 [[...], [...], ...]，没有分号。
    if len(chunks) < 2:
        chunks = re.findall(r"\[\s*([0-9\s,]+)\s*\]", raw)

    dims = []
    for chunk in chunks:
        values = [int(v) for v in re.findall(r"\d+", chunk)]
        if values:
            dims.append(values)

    if len(dims) < 2 or len(dims[0]) < 2 or len(dims[1]) < 2:
        return None

    M, K = dims[0][0], dims[0][1]
    N = dims[1][1]
    return M, K, N


def _parse_output_dtype(raw_dtype):
    """从 Output Data Types 中识别当前调用的输出类型。"""
    text = (raw_dtype or "").upper()
    if "INT32" in text:
        return "INT32"
    if "BF16" in text or "BFLOAT16" in text:
        return "BF16"
    return None


def parse_kernel_details(csv_path, test_shapes):
    """解析 kernel_details.csv 中 QmmCustom 每次调用的 Duration。

    CANN 8.5.x 的兼容 Stamp 路径可能只写入 Name/Duration，而不写
    Input Shapes 和 Output Data Types。此时不能直接索引 parts[1]，
    而是按本单元实际执行顺序回填：每个规格先 INT32、再 BF16。
    """
    candidates = []

    # utf-8-sig 同时兼容带 BOM 和不带 BOM 的 CSV。
    with open(csv_path, newline="", encoding="utf-8-sig", errors="replace") as f:
        reader = csv.DictReader(f)
        for source_index, row in enumerate(reader):
            name = (row.get("Name", "") or "").strip()
            op_type = (row.get("Type", "") or "").strip()
            canonical = re.sub(r"[^a-z0-9]", "", f"{name} {op_type}".lower())
            if "qmmcustom" not in canonical:
                continue

            duration_us = _safe_float(row.get("Duration(us)", "0"), 0.0)
            if duration_us <= 0:
                continue

            shape = _parse_input_shapes(row.get("Input Shapes", ""))
            dtype_tag = _parse_output_dtype(row.get("Output Data Types", ""))

            candidates.append({
                "source_index": source_index,
                "start_time": _safe_float(row.get("Start Time(us)", ""), float(source_index)),
                "shape": shape,
                "dtype": dtype_tag,
                "duration_us": duration_us,
                "name": name,
                "type": op_type,
            })

    # kernel_details.csv 通常已按时间排列；这里显式排序以兼容不同版本。
    candidates.sort(key=lambda item: (item["start_time"], item["source_index"]))

    expected_calls = []
    for M, K, N in test_shapes:
        expected_calls.append((M, K, N, "INT32"))
        expected_calls.append((M, K, N, "BF16"))

    if not candidates:
        return []

    expected_count = len(expected_calls)
    if len(candidates) > expected_count:
        print(
            f"提示：找到 {len(candidates)} 条 QmmCustom 记录，"
            f"本轮测试预期 {expected_count} 条；按时间取最后 {expected_count} 条。"
        )
        candidates = candidates[-expected_count:]
    elif len(candidates) < expected_count:
        print(
            f"警告：只找到 {len(candidates)} 条 QmmCustom 记录，"
            f"本轮测试预期 {expected_count} 条，缺失项将在汇总中显示为 '-'。"
        )

    rows = []
    fallback_count = 0
    for index, candidate in enumerate(candidates):
        # Shape/Dtype 元数据完整时优先使用 CSV；缺失时按调用顺序回填。
        expected = expected_calls[index] if index < expected_count else None
        if candidate["shape"] is not None:
            M, K, N = candidate["shape"]
        elif expected is not None:
            M, K, N = expected[:3]
            fallback_count += 1
        else:
            continue

        dtype_tag = candidate["dtype"]
        if dtype_tag is None and expected is not None:
            dtype_tag = expected[3]
            fallback_count += 1
        if dtype_tag is None:
            continue

        rows.append({
            "M": M,
            "K": K,
            "N": N,
            "dtype": dtype_tag,
            "duration_us": candidate["duration_us"],
        })

    if fallback_count:
        print(
            "说明：当前 Profiler 的 QmmCustom 记录未完整携带 Shape/Dtype，"
            "已依据测试循环顺序完成回填。"
        )

    return rows


def find_kernel_details_csv(prof_dir):
    """按文件修改时间获取本次最新生成的 kernel_details.csv。"""
    prof_path = Path(prof_dir)
    if not prof_path.exists():
        return None

    candidates = list(prof_path.glob("*/ASCEND_PROFILER_OUTPUT/kernel_details.csv"))
    if not candidates:
        return None

    latest = max(candidates, key=lambda p: p.stat().st_mtime_ns)
    return str(latest)


# ============================================================
# 主流程: 先 warmup, 再开启 profiler 采集, 最后汇总
# ============================================================
int32_results = []
bf16_results = []

# warmup: 不采集数据, 让 NPU 稳定
if WARMUP_ITERS > 0:
    print("=" * 60)
    print(f"Warmup: 运行 {WARMUP_ITERS} 轮不采集数据")
    print("=" * 60)
    for _ in range(WARMUP_ITERS):
        for M, K, N in TEST_SHAPES:
            test_int32(M, K, N)
            test_bf16(M, K, N)

# 本轮共执行 2 * len(TEST_SHAPES) 次被测调用。
# 显式 schedule 能在最后一次 step 后进入保存状态，避免 Stop 时仍处于 RECORD。
PROFILE_STEPS = 2 * len(TEST_SHAPES)
profile_schedule = torch_npu.profiler.schedule(
    wait=0,
    warmup=0,
    active=PROFILE_STEPS,
    repeat=1,
)

os.makedirs(PROF_OUTPUT_DIR, exist_ok=True)
profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    schedule=profile_schedule,
    experimental_config=torch_npu.profiler._ExperimentalConfig(
        profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
        aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
    ),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(PROF_OUTPUT_DIR),
)

profiler.start()

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))
    profiler.step()

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))
    profiler.step()

profiler.stop()

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")

# ============================================================
# Profiler 数据汇总: 每种规格每种 dtype 的 Duration(us)
# ============================================================
print()
print("=" * 70)
print("Profiler 数据汇总 (QmmCustom Duration)")
print("=" * 70)

csv_path = find_kernel_details_csv(PROF_OUTPUT_DIR)
print(f"原始数据路径：{csv_path}")
if csv_path:
    kernel_rows = parse_kernel_details(csv_path, TEST_SHAPES)
    if kernel_rows:
        shape_map = {}
        for kr in kernel_rows:
            key = (kr["M"], kr["K"], kr["N"])
            shape_map.setdefault(key, {})[kr["dtype"]] = kr["duration_us"]

        print(f"{'规格 (M,K,N)':>24} | {'INT32 Duration(us)':>18} | {'BF16 Duration(us)':>18}")
        print("-" * 70)
        for M, K, N in TEST_SHAPES:
            key = (M, K, N)
            int32_dur = shape_map.get(key, {}).get("INT32", "-")
            bf16_dur = shape_map.get(key, {}).get("BF16", "-")
            int32_str = f"{int32_dur:.3f}" if isinstance(int32_dur, float) else int32_dur
            bf16_str = f"{bf16_dur:.3f}" if isinstance(bf16_dur, float) else bf16_dur
            tag = f"M={M},K={K},N={N}"
            print(f"{tag:>24} | {int32_str:>18} | {bf16_str:>18}")
    else:
        print(f"在 {csv_path} 中未找到可解析的 QmmCustom 算子数据")
else:
    print(f"未在 {PROF_OUTPUT_DIR} 中找到 kernel_details.csv 文件")
    print("请检查 profiler 输出目录")

print("=" * 70)


[2026-07-23 15:29:04] [WARNING] [14143] profiler.py: Profiler won't be using warmup, this can skew profiler results
规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格4: M=1, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格5: M=50, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格6: M=50, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格7: M=50, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格8: M=50, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格9: M=4096, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格10: M=4096, K=40

---
## 6 自定义算子接入模型

### 6.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.ascendc_ops.qmm_custom` 算子。

查看原始实现：

In [7]:
!grep -n 'npu_quant_matmul' {RECIPE_ROOT}/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py

94:        # --- QmmCustom: replace npu_quant_matmul ---


### 6.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.ascendc_ops.qmm_custom`。

替换后的方法为：

```python
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight, 
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [8]:
# 6.2 替换实现: 将 torch_npu.npu_quant_matmul 替换为 torch.ops.ascendc_ops.qmm_custom

# 1: 导入libascendc_ops.so
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
print('[替换] 目标文件:', W8A8_FILE)

ct_code = W8A8_FILE.read_text(encoding='utf-8')
_lib_path = str(QMM_CUSTOM_DIR / "build/libascendc_ops.so")
_load_lib_stmt = 'torch.ops.load_library(r"' + _lib_path + '")'
if _load_lib_stmt not in ct_code:
    # 在 'import torch' 行后插入 'torch.ops.load_library'
    ct_code = ct_code.replace('import torch\n', 'import torch\n' + _load_lib_stmt + '\n')
    W8A8_FILE.write_text(ct_code, encoding='utf-8')
    print('[插入] compressed_tensors.py: 已添加 torch.ops.load_library')
else:
    print('[插入] compressed_tensors.py: torch.ops.load_library 已存在，跳过')

# 2: 在 compressed_tensors_w8a8_int8.py 中替换 npu_quant_matmul → qmm_custom
code = W8A8_FILE.read_text(encoding='utf-8')

# 替换 npu_quant_matmul 调用为 qmm_custom 调用
old_call = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
new_call = (
    '# --- 替换核心: npu_quant_matmul → qmm_custom ---\n'
    '        x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale)\n'
    '        # --- 替换结束 ---'
)

if old_call in code:
    code = code.replace(old_call, new_call)
    W8A8_FILE.write_text(code, encoding='utf-8')
    print('[替换] 成功: npu_quant_matmul → qmm_custom')
else:
    print('[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换')

# 验证替换结果
result = W8A8_FILE.read_text(encoding='utf-8')
if 'torch.ops.ascendc_ops.qmm_custom' in result:
    print('[验证] 替换已生效，当前使用 qmm_custom 算子')
else:
    print('[验证] 替换未生效，请检查文件内容')


[替换] 目标文件: /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py
[插入] compressed_tensors.py: torch.ops.load_library 已存在，跳过
[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换
[验证] 替换已生效，当前使用 qmm_custom 算子


### 6.3 替换实现后执行量化推理

指定本地的量化模型权重路径，调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，对比替换前的生成文本是否一致。

**注意：需要先完成第五章的量化模型推理过程，并将`W8A8_WEIGHT`指定为本地的量化模型权重的路径**

In [9]:
import sys, subprocess, os
from inference_scripts.recipe_workflow import prepare_runtime_yaml

# 将W8A8_WEIGHT设置为量化模型权重实际所在路径
W8A8_WEIGHT = TUTORIAL_DIR / 'src/data/models/Qwen3-8B-W8A8'

YAML_A8W8_TEMPLATE =  RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8_CUSTOM  = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom.yaml'
prepare_runtime_yaml(YAML_A8W8_TEMPLATE, YAML_A8W8_CUSTOM, model_path=str(W8A8_WEIGHT))

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom.yaml)
world_size = 1
[INFO][2026-07-23 15:29:28.661][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-23 15:29:28.661][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-23 15:29:28.662][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-23 15:29:28.664][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-23 15:29:28.664][patch_getenv.py:15] get env RES_PATH = res/20260723/qwen3_8b_qwen3_8b_a8w8_custom
[INFO][2026-07-23 15:29:28.664][infer.py:146] Inference Configuration
[INFO][2026-07-23 15:29:28.664][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_

---
## 7 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 7.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [10]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_A8W8_CUSTOM_PROFILER = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml'

prepare_profiling_yaml(YAML_A8W8_CUSTOM, YAML_A8W8_CUSTOM_PROFILER, enable_profiler=True)

{'model_config': {'model_name': 'qwen3_8b',
  'model_path': '/mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8',
  'exe_mode': 'eager',
  'enable_profiler': True,
  'with_ckpt': True,
  'enable_cache_compile': False,
  'enable_static_kernel': False},
 'data_config': {'dataset': 'default', 'input_truncated_len': 4096},
 'parallel_config': {'world_size': 1,
  'attn_tp_size': 1,
  'moe_tp_size': 1,
  'embed_tp_size': 1,
  'lmhead_tp_size': 1},
 'scheduler_config': {'max_new_tokens': 256, 'batch_size': 1}}

### 7.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [11]:
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM_PROFILER],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml)
world_size = 1
[INFO][2026-07-23 15:30:34.490][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-23 15:30:34.490][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-23 15:30:34.490][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-23 15:30:34.493][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-23 15:30:34.493][patch_getenv.py:15] get env RES_PATH = res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler
[INFO][2026-07-23 15:30:34.493][infer.py:146] Inference Configuration
[INFO][2026-07-23 15:30:34.493][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/cann-learning-hub/tutorials/ll

### 7.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [12]:
import pandas as pd
from pathlib import Path

RES_DIR = RECIPE_ROOT / 'models/qwen/res'

# 取最新的 profiling 结果目录
result_dirs = sorted([d for d in RES_DIR.iterdir() if d.is_dir()], key=lambda d: d.name, reverse=True)
latest_result = result_dirs[0]
print(f'最新结果目录: {latest_result}')

def find_kernel_csv(phase_dir):
    matches = sorted(phase_dir.rglob('kernel_details.csv'), key=lambda p: p.parent.parent.name, reverse=True)
    return matches[0] if matches else None

prefill_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill')
decode_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode')
assert prefill_csv and decode_csv, 'kernel_details.csv 未找到，请确认 profiling 已完成'
print(f'Prefill CSV: {prefill_csv}')
print(f'Decode CSV: {decode_csv}')

# 读取 CSV 并筛选 QmmCustom 算子
prefill_df = pd.read_csv(prefill_csv)
decode_df = pd.read_csv(decode_csv)

prefill_qmm = prefill_df[prefill_df['Type'].str.contains('QmmCustom')].copy()
decode_qmm = decode_df[decode_df['Type'].str.contains('QmmCustom')].copy()

# 按 Input Shapes 分组，统计每种 shape 下的平均 Duration
def qmm_duration_stats(df, phase):
    stats = df.groupby(['Input Shapes', 'Output Data Types'])['Duration(us)'].agg(['mean', 'count']).reset_index()
    stats.columns = ['Input Shapes', 'Output Dtype', 'Avg Duration(us)', 'Calls']
    stats = stats.sort_values('Avg Duration(us)', ascending=False).reset_index(drop=True)
    stats.insert(0, 'Phase', phase)
    return stats

prefill_stats = qmm_duration_stats(prefill_qmm, 'Prefill')
decode_stats = qmm_duration_stats(decode_qmm, 'Decode')

combined = pd.concat([prefill_stats, decode_stats], ignore_index=True)
display(combined.style.set_caption('QmmCustom 各 Shape 平均耗时 (us)'))

最新结果目录: /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723
Prefill CSV: /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill/b83ad510f83242d2b1992b1dc7da0a3c_26530_20260723153102599_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
Decode CSV: /mnt/workspace/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode/b83ad510f83242d2b1992b1dc7da0a3c_26530_20260723153106369_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv


,Phase,Input Shapes,Output Dtype,Avg Duration(us),Calls


## 8 提交要求

> 完整的提交说明参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 性能数据收集成功。
3. **文件命名**：`gitcode账号_result.ipynb`。
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）。
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译和单算子功能性能测试，并将该自定义算子接入了 Qwen3-8B 量化模型，验证功能并观测了其在模型中的性能。


### 附录
实践相关参考材料链接：
- [Ascend C 高阶api文档](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%AB%98%E9%98%B6API/%E9%AB%98%E9%98%B6API.md)
- [昇腾社区文档](https://www.hiascend.com/zh/document)
- [Ascend C 算子开发系列教程](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development)
- [Ascend C 算子功能调试](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development/07_Troubleshooting)
- [Matmul最佳实践样例](https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance/README.md)
- [CANN内置量化矩阵乘算子实现](https://gitcode.com/cann/ops-nn/tree/master/matmul/quant_batch_matmul_v3)